# 03. Adding Azure Monitor OpenTelemetry Tracing to a LangChain Agent

This notebook takes the exact same CloudXeus tool-calling agent from notebook 02 (`cloudxeus_agent.py`) and adds distributed tracing: every model call and tool call gets emitted as an OpenTelemetry span to an **Azure Monitor Application Insights** resource, via `langchain_azure_ai`'s `AzureAIOpenTelemetryTracer` callback. It belongs to Section 04, "Agent frameworks on top of Azure AI Foundry."

**Difficulty: Advanced**

The tracer callback and LangChain's `.with_config({"callbacks": [...]})` mechanism are LangChain framework concepts and are **not** AI-102 exam material. What *is* exam-relevant is the destination of the telemetry — Azure Monitor / Application Insights — which is a first-class AI-102 topic under "monitor AI solutions" (the exam expects you to know how to wire Azure AI resources to Application Insights for observability, regardless of which client library produced the calls).

## Prerequisites

**Pip packages:**
```bash
pip install langchain langchain-openai langchain-azure-ai python-dotenv
```
`langchain-azure-ai` is the package that provides `langchain_azure_ai.callbacks.tracers.AzureAIOpenTelemetryTracer` — it is not currently in the repo root `requirements.txt`. (The original script's comment `# pip install -U microsoft-opentelemetry` refers to an OpenTelemetry exporter dependency that `langchain-azure-ai` pulls in transitively; installing `langchain-azure-ai` itself is the actual requirement for the import used below.)

**Azure resources required:**
- The same Azure AI Foundry project + model deployment as notebooks 01–02.
- An **Application Insights** resource (or a Log Analytics-backed Azure Monitor workspace) to receive the traces. Many Azure AI Foundry projects have one connected by default; you can also create a standalone one.

**Environment variables** (add to root `.env`):
```
AZURE_AI_OPENAI_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/openai/v1
AZURE_AI_OPENAI_API_KEY=<your-foundry-api-key>
AZURE_AI_MODEL_DEPLOYMENT=<your-chat-deployment-name>
APPLICATIONINSIGHTS_CONNECTION_STRING=<your-application-insights-connection-string>
```
Note the original script hardcoded a real-looking Application Insights connection string (including an instrumentation key GUID) directly in source — that has been replaced below with `os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")`. **Never commit a real connection string to source control**; find yours under your Application Insights resource → Overview → Connection String in the Azure Portal.

## What You'll Learn

- How to attach an OpenTelemetry tracer to a LangChain agent via `.with_config({"callbacks": [...]})`
- What `AzureAIOpenTelemetryTracer` captures (model calls, tool calls, latencies) and where it sends that data
- The distinction between `enable_content_recording` (whether prompt/response *text* is captured, not just metadata) and basic span tracing
- Why observability wiring is framework-agnostic even though this example happens to use LangChain

### Step 1 — Imports, configuration, and the base model + tools

This is identical to notebook 02: build the `ChatOpenAI` client against the Foundry endpoint, and define the same two tools (`get_order_status`, `get_inventory`). The only new import is `AzureAIOpenTelemetryTracer`.

💡 **Framework note:** nothing about model/tool setup changes when you add tracing — instrumentation is applied *around* the agent (via callbacks), not baked into how the agent itself is built. This mirrors how you'd instrument any Azure SDK client: the client code doesn't change, you attach a monitoring layer alongside it.

🔄 **Alternatives:** rather than a LangChain-specific tracer, you could instrument the same Foundry endpoint calls with the vendor-neutral `opentelemetry-instrumentation-openai` / Azure Monitor OpenTelemetry Distro directly against the `openai` SDK — useful if you want tracing that doesn't depend on LangChain at all.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_azure_ai.callbacks.tracers import AzureAIOpenTelemetryTracer

load_dotenv()

endpoint = os.getenv("AZURE_AI_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT")
api_key = os.getenv("AZURE_AI_OPENAI_API_KEY")

model = ChatOpenAI(
    base_url=endpoint,
    api_key=api_key,
    model=deployment_name,
)


In [ ]:
@tool
def get_order_status(order_id: str) -> str:
    """Get the current status of a CloudXeus order by order ID."""
    orders = {
        "ORD-001": "Dispatched — arriving tomorrow.",
        "ORD-002": "Processing — not yet shipped.",
        "ORD-003": "Delivered on June 12, 2026.",
    }
    return orders.get(order_id, f"Order {order_id} not found.")


@tool
def get_inventory(product_id: str) -> str:
    """Check the available inventory for a CloudXeus product by product ID."""
    inventory = {
        "PRD-A1": "142 units in stock.",
        "PRD-B2": "0 units — out of stock.",
        "PRD-C3": "37 units in stock.",
    }
    return inventory.get(product_id, f"Product {product_id} not found.")


### Step 2 — Configure the OpenTelemetry tracer

`OTEL_SERVICE_NAME` is the label this app's traces will show up under in Application Insights when multiple apps send telemetry to the same resource — set it distinctly per app so you can filter by service in queries. `AzureAIOpenTelemetryTracer` is constructed with the Application Insights connection string, a human-readable `name`, an `agent_id`, and `enable_content_recording=True` (which additionally captures prompt/response text, not just latency/token metadata — turn this off in production if prompts may contain sensitive data).

💡 **Exam tip:** `enable_content_recording` maps directly to an AI-102 concern — Azure AI Foundry's own built-in tracing has the same on/off toggle for content capture, because logging full prompts/responses has data-residency and compliance implications. Know that "trace metadata" and "trace content" are separable settings on the exam.

🔄 **Alternatives:** Azure AI Foundry Agent Service has built-in tracing you can enable directly on a persisted Agent resource (see `11_azure_ai_foundry/`) without touching LangChain at all — that's the more "exam-native" path when you're using the Agents/Threads/Runs API rather than a LangChain agent.

In [ ]:
OTEL_SERVICE_NAME = "cloudxeus-langchain-ops-agent"
appinsights_connection_string = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")

tracer = AzureAIOpenTelemetryTracer(
    connection_string=appinsights_connection_string,
    name="CloudXeus LangChain Ops Agent",
    agent_id="cloudxeus-langchain-ops-agent",
    enable_content_recording=True,
)


### Step 3 — Build the agent and attach the tracer

`create_agent(...)` builds the same agent as notebook 02. `.with_config({"callbacks": [tracer]})` returns a new runnable that behaves identically but additionally emits telemetry through the tracer on every step — model call, tool call, and the final response.

💡 **Framework note:** `.with_config` is LangChain's general mechanism for attaching cross-cutting concerns (callbacks, tags, run names) to any runnable without changing its core logic — the same mechanism works for chains, retrievers, and graphs, not just agents.

🔄 **Alternatives:** callbacks can also be passed per-call (`agent.invoke(..., config={"callbacks": [tracer]})`) instead of baked into the runnable via `.with_config`, which is useful if you only want tracing on some invocations.

In [ ]:
agent = create_agent(
    model=model,
    tools=[get_order_status, get_inventory],
    system_prompt=(
        "You are a helpful CloudXeus operations assistant. "
        "Use the available tools to answer questions accurately."
    ),
).with_config(
    {"callbacks": [tracer]}
)


### Step 4 — Invoke the agent

Same invocation pattern as notebook 02. The difference is invisible in the printed output — it shows up as spans in your Application Insights resource (under Transaction Search / Application Map / Logs), not in the notebook.

💡 **Exam tip:** on AI-102, know that Application Insights ties into Azure Monitor's broader alerting/dashboarding story — traces like these can drive alerts on latency or error-rate regressions in a production AI agent, which is the operational payoff of instrumenting calls this way.

🔄 **Alternatives:** for local debugging without a real Application Insights resource, you can point OpenTelemetry at a local collector (e.g., the OpenTelemetry Collector or Jaeger) instead — useful during development before wiring up the real Azure resource.

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "What is the status of order ORD-002 and "
                    "how many units of PRD-A1 do we have?"
                ),
            }
        ]
    }
)

print(response["messages"][-1].content)


## Summary

This notebook re-ran the CloudXeus tool-calling agent from notebook 02, unchanged in behavior, but attached an `AzureAIOpenTelemetryTracer` callback so every model and tool call is exported as an OpenTelemetry span to Application Insights. The printed output is identical to notebook 02 — the value here is entirely in what now shows up in Azure Monitor: latency per step, which tools were called, and (because `enable_content_recording=True`) the actual prompt/response content for debugging.

## Try It Yourself

1. **Easy:** Set `enable_content_recording=False` and reason about what would be missing from your traces afterward (latency/counts still present, but not prompt/response text) — useful for a compliance-sensitive deployment.
2. **Intermediate:** After a real run, go to your Application Insights resource → Transaction Search and find the trace for this notebook's invocation using the `agent_id` value as a filter.
3. **Advanced:** Add a second tracer/service name for a *different* agent (e.g., copy notebook 02 as a second "billing" agent with its own `OTEL_SERVICE_NAME`) sending telemetry to the same Application Insights resource, then use Log Analytics queries to compare their latency distributions.